# Full Day RFI Flagging

**by Josh Dillon**, last updated August 27, 2026

This notebook synthesizes a whole night of the per-file z-scores produced by
[file_sky_calibration](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/phase_II/file_sky_calibration.ipynb)
to find and flag low-level RFI. That notebook sky-calibrates each file independently, redundantly
averages the calibrated data with inverse-variance weights, performs a high-pass delay filter, and
incoherently averages across baselines, creating a per-polarization z-score waterfall. This notebook
takes the whole night of those z-scores and finds a new set of flags.

Unlike its H6C predecessor
([full_day_rfi_round_2](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/full_day_rfi_round_2.ipynb)),
it runs *before* calibration smoothing — the z-scores come from per-file calibration, not smoothed
calibration — so it modifies no calibration files. Its only outputs are per-file `UVFlag` waterfall
`*.flag_waterfall.h5` files for downstream consumption; the a posteriori yaml summarizing the day's
final flags (including `ex_ants`) is calibration smoothing's job, as the last stage that touches
flags. The H6C-era 2D DPSS filtering of whole-day autocorrelations has been deliberately dropped;
protecting calibration smoothing from temporally-structured, spectrally-broadband contamination is
planned as robust rejection on the gain waterfalls themselves.

Here's a set of links to skip to particular figures and tables:

• [Figure 1: Waterfall of Maximum z-Score of Either Polarization Before Full-Day Flagging](#Figure-1:-Waterfall-of-Maximum-z-Score-of-Either-Polarization-Before-Full-Day-Flagging)

• [Figure 2: Histogram of z-scores](#Figure-2:-Histogram-of-z-scores)

• [Figure 3: Waterfall of Maximum z-Score of Either Polarization After Full-Day Flagging](#Figure-3:-Waterfall-of-Maximum-z-Score-of-Either-Polarization-After-Full-Day-Flagging)

• [Figure 4: Spectra of Time-Averaged z-Scores](#Figure-4:-Spectra-of-Time-Averaged-z-Scores)

• [Figure 5: Summary of Flags Before and After Full-Day Flagging](#Figure-5:-Summary-of-Flags-Before-and-After-Full-Day-Flagging)

In [ ]:
import time
tstart = time.time()
!hostname
!date

In [ ]:
import os
import toml
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import h5py
import hdf5plugin  # REQUIRED to have the compression plugins available
import numpy as np
import glob
import matplotlib.pyplot as plt
import matplotlib
import copy
import warnings
from pyuvdata import UVFlag
from hera_cal import utils
from hera_qm import xrfi
from hera_qm.time_series_metrics import true_stretches
from hera_filters import dspec

from IPython.display import display, HTML
%matplotlib inline
display(HTML("<style>.container { width:100% !important; }</style>"))
_ = np.seterr(all='ignore')  # get rid of red warnings
%config InlineBackend.figure_format = 'retina'

In [ ]:
# parse wrapper-level environment variables: paths, plus the save toggle
SUM_FILE = os.environ.get("SUM_FILE", None)
# SUM_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/2459935/zen.2459935.21341.sum.uvh5'
TOML_FILE = os.environ.get("TOML_FILE", None)
# TOML_FILE = '/lustre/aoc/projects/hera/phase-II-analysis/idr1/src/hera_pipelines/pipelines/phase_II/idr1/v1/analysis/phase_II_analysis.toml'
SAVE_RESULTS = os.environ.get("SAVE_RESULTS", "TRUE").upper() == "TRUE"
# which [WorkFlow] action is running this notebook
ACTION = os.environ.get("ACTION", "FULL_DAY_RFI_NOTEBOOK")

# default suffixes for the files this notebook reads or writes
SUM_SUFFIX = 'sum.uvh5'
RED_AVG_ZSCORE_SUFFIX = 'sum.red_avg_zscore.h5'
FLAG_WATERFALL_SUFFIX = 'sum.flag_waterfall.h5'

# default settings, overridden by the TOML's [FULL_DAY_RFI_OPTS] section (if given)
Z_THRESH = 4.0
WS_Z_THRESH = 2.0
AVG_Z_THRESH = 1.0
MAX_FREQ_FLAG_FRAC = 0.25
MAX_TIME_FLAG_FRAC = 0.1
AVG_SPECTRUM_FILTER_DELAY = 250.0  # in ns
EIGENVAL_CUTOFF = 1e-12
TIME_AVG_DELAY_FILT_SNR_THRESH = 4.0
TIME_AVG_DELAY_FILT_SNR_DYNAMIC_RANGE = 1.5

toml_options = (toml.load(TOML_FILE) if TOML_FILE is not None else {})

# Take the suffixes this notebook reads or writes from [DATA_PRODUCTS], scoped by that
# section's produced_by/consumed_by wiring, and require that wiring to agree exactly with
# the defaults above. That way the arrows drawn on the pipeline flowchart cannot drift from
# what this notebook actually touches: adding an edge there without using the file here (or
# vice versa) fails loudly, in the first cell, rather than silently.
declared_suffixes = {name for name in list(globals()) if name.endswith('_SUFFIX')}
wired_suffixes = set()
for product, spec in toml_options.get('DATA_PRODUCTS', {}).items():
    producers, consumers = (spec.get(key, []) for key in ['produced_by', 'consumed_by'])
    producers = ([producers] if isinstance(producers, str) else producers)
    consumers = ([consumers] if isinstance(consumers, str) else consumers)
    if 'suffix' in spec and (ACTION in producers or ACTION in consumers):
        wired_suffixes.add(f'{product.upper()}_SUFFIX')
        globals()[f'{product.upper()}_SUFFIX'] = spec['suffix']
if toml_options:
    assert wired_suffixes == declared_suffixes, (
        f'[DATA_PRODUCTS] wires {sorted(wired_suffixes)} to {ACTION}, '
        f'but this notebook declares {sorted(declared_suffixes)}.')

if 'FULL_DAY_RFI_OPTS' in toml_options:
    print(f'Loading overrides from [FULL_DAY_RFI_OPTS] in {TOML_FILE}.')
    for key, val in toml_options['FULL_DAY_RFI_OPTS'].items():
        globals()[key.upper()] = val

# build globs and output paths
sum_glob = '.'.join(SUM_FILE.split('.')[:-3]) + '.*.' + SUM_SUFFIX
zscore_glob = sum_glob.replace(SUM_SUFFIX, RED_AVG_ZSCORE_SUFFIX)

for setting in ['SUM_FILE', 'TOML_FILE', 'SAVE_RESULTS', 'ACTION', 'SUM_SUFFIX', 'RED_AVG_ZSCORE_SUFFIX',
                'FLAG_WATERFALL_SUFFIX', 'Z_THRESH', 'WS_Z_THRESH', 'AVG_Z_THRESH',
                'MAX_FREQ_FLAG_FRAC', 'MAX_TIME_FLAG_FRAC', 'AVG_SPECTRUM_FILTER_DELAY', 'EIGENVAL_CUTOFF',
                'TIME_AVG_DELAY_FILT_SNR_THRESH', 'TIME_AVG_DELAY_FILT_SNR_DYNAMIC_RANGE']:
    print(f'{setting} = {eval(setting)}')

# Load z-scores

In [ ]:
# load z-scores
zscore_files = sorted(glob.glob(zscore_glob))
print(f'Found {len(zscore_files)} *.{RED_AVG_ZSCORE_SUFFIX} files starting with {zscore_files[0]}.')
uvf = UVFlag(zscore_files)

In [ ]:
# extract z-scores and correct by a single number per polarization to account for biases created by filtering
x_orientation = uvf.telescope.get_x_orientation_from_feeds()
zscore = {pol: uvf.metric_array[:, :, np.argwhere(uvf.polarization_array == utils.polstr2num(pol, x_orientation=x_orientation))[0][0]] for pol in ['ee', 'nn']}
zscore = {pol: zscore[pol] - np.nanmedian(zscore[pol]) for pol in zscore}

In [ ]:
freqs = uvf.freq_array
times = uvf.time_array
dt = np.median(np.diff(times))

In [ ]:
extent = [freqs[0] / 1e6, freqs[-1] / 1e6, times[-1] - int(times[0]), times[0] - int(times[0])]
lst_grid = utils.JD2LST(times) * 12 / np.pi
lst_grid[lst_grid > lst_grid[-1]] -= 24

In [ ]:
def plot_max_z_score(zscore, flags=None, vmin=-5, vmax=5):
    if flags is None:
        flags = np.any(~np.isfinite(list(zscore.values())), axis=0)
    fig, ax = plt.subplots(figsize=(14,10), dpi=100)
    im = ax.imshow(np.where(flags, np.nan, np.nanmax([zscore['ee'], zscore['nn']], axis=0)), aspect='auto', 
               cmap='coolwarm', interpolation='none', vmin=vmin, vmax=vmax, extent=extent)
    plt.colorbar(im, ax=ax, location='top', label='Max z-score of either polarization', extend='both', aspect=40, pad=.02)
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel(f'JD - {int(times[0])}')

    # Add LST right axis
    ax2 = ax.twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    plt.tight_layout()

# *Figure 1: Waterfall of Maximum z-Score of Either Polarization Before Full-Day Flagging*

Shows the worse of the two results from
[delay_filtered_average_zscore](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/delay_filtered_average_zscore.ipynb)
from either polarization. Dips near flagged channels are expected, due to overfitting of noise.
Positive-going excursions are problematic and likely evidence of RFI.

In [ ]:
plot_max_z_score(zscore)

In [ ]:
def plot_histogram():
    plt.figure(figsize=(14,4), dpi=100)
    bins = np.arange(-50, 100, .1)
    hist_ee = plt.hist(np.ravel(zscore['ee']), bins=bins, density=True, label='ee-polarized z-scores', alpha=.5)
    hist_nn = plt.hist(np.ravel(zscore['nn']), bins=bins, density=True, label='nn-polarized z-scores', alpha=.5)
    plt.plot(bins, (2*np.pi)**-.5 * np.exp(-bins**2 / 2), 'k:', label='Gaussian approximate\nnoise-only distribution')
    plt.axvline(WS_Z_THRESH, c='r', ls='--', label='Watershed z-score')
    plt.axvline(Z_THRESH, c='r', ls='-', label='Threshold z-score')
    plt.yscale('log')
    all_densities = np.concatenate([hist_ee[0][hist_ee[0] > 0], hist_nn[0][hist_nn[0] > 0]]) 
    plt.ylim(np.min(all_densities) / 2, np.max(all_densities) * 2)
    plt.xlim([-50, 100])
    plt.legend()
    plt.xlabel('z-score')
    plt.ylabel('Density')
    plt.tight_layout()

# *Figure 2: Histogram of z-scores*

Shows a comparison of the histogram of z-scores in this file (one per polarization) to a Gaussian
approximation of what one might expect from thermal noise. Without filtering, the actual
distribution is a weighted sum of Rayleigh distributions. Filtering further complicates this. To
make the z-scores more reliable, a single per-polarization median is subtracted from each waterfall,
which allows us to flag low-level outliers with more confidence. Any points beyond the solid red
line are flagged. Any points neighboring a flag beyond the dashed red line are also flagged.
Finally, flagging is performed for low-level outliers in whole times or channels.

In [ ]:
plot_histogram()

## Perform flagging

In [ ]:
def iteratively_flag_on_averaged_zscore(flags, zscore, avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True):
    '''Flag whole integrations or channels based on average z-score. This is done
    iteratively to prevent bad times affecting channel averages or vice versa.'''
    flagged_chan_count = 0
    flagged_int_count = 0
    while True:
        zspec = avg_func(np.where(flags, np.nan, zscore), axis=0)
        ztseries = avg_func(np.where(flags, np.nan, zscore), axis=1)

        if (np.nanmax(zspec) < avg_z_thresh) and (np.nanmax(ztseries) < avg_z_thresh):
            break

        if np.nanmax(zspec) >= np.nanmax(ztseries):
            flagged_chan_count += np.sum((zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh))
            flags[:, (zspec >= np.nanmax(ztseries)) & (zspec >= avg_z_thresh)] = True
        else:
            flagged_int_count += np.sum((ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh))
            flags[(ztseries >= np.nanmax(zspec)) & (ztseries >= avg_z_thresh), :] = True

    if verbose:
        print(f'\tFlagging an additional {flagged_int_count} integrations and {flagged_chan_count} channels.')

def impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True):
    '''Flag channels already flagged more than max_flag_frac (excluding completely flagged times).'''
    unflagged_times = ~np.all(flags, axis=1)
    frequently_flagged_chans =  np.mean(flags[unflagged_times, :], axis=0) >= max_flag_frac
    if verbose:
        print(f'\tFlagging {np.sum(frequently_flagged_chans) - np.sum(np.all(flags, axis=0))} channels previously flagged {max_flag_frac:.2%} or more.')        
    flags[:, frequently_flagged_chans] = True 
        
def impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True):
    '''Flag times already flagged more than max_flag_frac (excluding completely flagged channels).'''
    unflagged_chans = ~np.all(flags, axis=0)
    frequently_flagged_times =  np.mean(flags[:, unflagged_chans], axis=1) >= max_flag_frac
    if verbose:
        print(f'\tFlagging {np.sum(frequently_flagged_times) - np.sum(np.all(flags, axis=1))} times previously flagged {max_flag_frac:.2%} or more.')
    flags[frequently_flagged_times, :] = True

def time_avg_zscore_dly_filt_SNRs(flags, filter_delay=AVG_SPECTRUM_FILTER_DELAY, eigenval_cutoff=EIGENVAL_CUTOFF):
    """Produces SNRs after time-averaging z-scores and delay filtering, accounting for flagging's effect on the filter."""
    # figure out high and low band based on FM gap at 100 MHz
    flagged_stretches = true_stretches(np.all(flags, axis=0))
    FM_gap = [fs for fs in flagged_stretches if fs.start <= np.argmin(np.abs(freqs - 100e6)) < fs.stop][0]
    low_band = slice((0 if flagged_stretches[0].start != 0 else flagged_stretches[0].stop), FM_gap.start)
    high_band = slice(FM_gap.stop, (len(freqs) if flagged_stretches[-1].stop != len(freqs) else flagged_stretches[-1].start))
    
    filt_SNR = {}
    for pol in zscore:
        # calculate timeavg_SNR and filter
        noise_prediction = 1.0 / np.sum(~flags, axis=0)**.5
        timeavg_SNR = np.nanmean(np.where(flags, np.nan, zscore[pol] / noise_prediction), axis=0) 
        wgts = np.where(np.isfinite(timeavg_SNR), 1, 0)
        model = np.zeros_like(timeavg_SNR)
        for band in [low_band, high_band]:
            filter_kwargs = dict(filter_centers=[0], filter_half_widths=[AVG_SPECTRUM_FILTER_DELAY / 1e9],
                                 eigenval_cutoff=[EIGENVAL_CUTOFF], suppression_factors=[EIGENVAL_CUTOFF])
            model[band], _, info = dspec.fourier_filter(freqs[band], np.where(np.isfinite(timeavg_SNR[band]), timeavg_SNR[band], 0),
                                                        wgts[band], mode="dpss_leastsq", **filter_kwargs)
            # Fall back to dpss_matrix for any skipped axes
            for i, s in info['status']['axis_0'].items():
                if s == 'skipped':
                    model[band], _, _ = dspec.fourier_filter(freqs[band], np.where(np.isfinite(timeavg_SNR[band]), timeavg_SNR[band], 0),
                                                             wgts[band], mode="dpss_matrix", **filter_kwargs)
        filt_SNR[pol] = timeavg_SNR - model

        # correct for impact of filter
        correction_factors = np.ones_like(wgts) * np.nan
        for band in [low_band, high_band]:
            X = dspec.dpss_operator(freqs[band], [0], filter_half_widths=[AVG_SPECTRUM_FILTER_DELAY / 1e9], eigenval_cutoff=[EIGENVAL_CUTOFF])[0]
            W = wgts[band]
            P = np.linalg.pinv(np.dot(X.T * W, X), hermitian=True) @ (X.T * W)
            leverage = np.real(np.sum(X * P.T, axis=1))
            correction = (1 - leverage)**.5
            correction_factors[band] = np.where(np.isfinite(correction) & (leverage > 0) & (leverage < 1), correction, np.nan)
        filt_SNR[pol] /= correction_factors
        # Flag channels with non-finite SNR (e.g. from degenerate leverage corrections)
        newly_flagged_chans = ~np.isfinite(filt_SNR[pol])
        if np.any(newly_flagged_chans):
            flags[:, newly_flagged_chans] = True
    
    return filt_SNR

def iteratively_flag_on_delay_filtered_time_avg_zscore(flags, thresh=TIME_AVG_DELAY_FILT_SNR_THRESH, dynamic_range=TIME_AVG_DELAY_FILT_SNR_DYNAMIC_RANGE,
                                                       filter_delay=AVG_SPECTRUM_FILTER_DELAY, eigenval_cutoff=EIGENVAL_CUTOFF):
    """Flag whole channels based on their outlierness after delay-filterd time-averaged zscores.
    This is done iteratively since the delay filter can be unduly influenced by large outliers."""
    filt_SNR = time_avg_zscore_dly_filt_SNRs(flags, filter_delay=AVG_SPECTRUM_FILTER_DELAY, eigenval_cutoff=EIGENVAL_CUTOFF)
    while True:
        largest_SNR = np.nanmax(list(filt_SNR.values()))
        if largest_SNR < thresh:
            break
        # 
        cut = np.max([thresh, largest_SNR / dynamic_range])
        for pol in filt_SNR:
            flags[:, filt_SNR[pol] > cut] = True
        filt_SNR = time_avg_zscore_dly_filt_SNRs(flags, filter_delay=AVG_SPECTRUM_FILTER_DELAY, eigenval_cutoff=EIGENVAL_CUTOFF)

In [ ]:
flags = np.any(~np.isfinite(list(zscore.values())), axis=0)
print(f'{np.mean(flags):.3%} of waterfall flagged to start.')

# flag whole integrations or channels using outliers in median
while True:
    nflags = np.sum(flags)
    for pol in ['ee', 'nn']:    
        iteratively_flag_on_averaged_zscore(flags, zscore[pol], avg_func=np.nanmedian, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags) == nflags:
        break  
print(f'{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with median z > {AVG_Z_THRESH}.')

# flag largest outliers
for pol in ['ee', 'nn']:
    flags |= (zscore[pol] > Z_THRESH) 
print(f'{np.mean(flags):.3%} of waterfall flagged after flagging z > {Z_THRESH} outliers.')
    
# watershed flagging
while True:
    nflags = np.sum(flags)
    for pol in ['ee', 'nn']:
        flags |= xrfi._ws_flag_waterfall(zscore[pol], flags, WS_Z_THRESH)
    if np.sum(flags) == nflags:
        break
print(f'{np.mean(flags):.3%} of waterfall flagged after watershed flagging on z > {WS_Z_THRESH} neighbors of prior flags.')

# flag whole integrations or channels using outliers in mean
while True:
    nflags = np.sum(flags)
    for pol in ['ee', 'nn']:    
        iteratively_flag_on_averaged_zscore(flags, zscore[pol], avg_func=np.nanmean, avg_z_thresh=AVG_Z_THRESH, verbose=True)
        impose_max_chan_flag_frac(flags, max_flag_frac=MAX_FREQ_FLAG_FRAC, verbose=True)
        impose_max_time_flag_frac(flags, max_flag_frac=MAX_TIME_FLAG_FRAC, verbose=True)
    if np.sum(flags) == nflags:
        break  
print(f'{np.mean(flags):.3%} of waterfall flagged after flagging whole times and channels with average z > {AVG_Z_THRESH}.')

# flag channels based on delay filter
iteratively_flag_on_delay_filtered_time_avg_zscore(flags, thresh=TIME_AVG_DELAY_FILT_SNR_THRESH, dynamic_range=TIME_AVG_DELAY_FILT_SNR_DYNAMIC_RANGE,
                                                   filter_delay=AVG_SPECTRUM_FILTER_DELAY, eigenval_cutoff=EIGENVAL_CUTOFF)
print(f'{np.mean(flags):.3%} of flagging channels that are {TIME_AVG_DELAY_FILT_SNR_THRESH}σ outliers after delay filtering the time average.')

# watershed flagging again
while True:
    nflags = np.sum(flags)
    for pol in ['ee', 'nn']:
        flags |= xrfi._ws_flag_waterfall(zscore[pol], flags, WS_Z_THRESH)
    if np.sum(flags) == nflags:
        break
print(f'{np.mean(flags):.3%} of waterfall flagged after another round of watershed flagging on z > {WS_Z_THRESH} neighbors of prior flags.')

## Show results of flagging

# *Figure 3: Waterfall of Maximum z-Score of Either Polarization After Full-Day Flagging*

The same as Figure 1, but after the flagging performed in this notebook.

In [ ]:
plot_max_z_score(zscore, flags=flags)

In [ ]:
def zscore_spectra(ylim=[-3, 3], flags=flags):
    fig, axes = plt.subplots(2, 1, figsize=(14,6), dpi=100, sharex=True, sharey=True, gridspec_kw={'hspace': 0})
    for ax, pol in zip(axes, ['ee', 'nn']):

        ax.plot(freqs / 1e6, np.nanmean(zscore[pol], axis=0),'r', label=f'{pol}-Polarization Before Full-Day Flagging', lw=.5)
        ax.plot(freqs / 1e6, np.nanmean(np.where(flags, np.nan, zscore[pol]), axis=0), label=f'{pol}-Polarization After Full-Day Flagging')
        ax.legend(loc='lower right')
        ax.set_ylabel('Time-Averged Z-Score\n(Excluding Flags)')
        ax.set_ylim(ylim)
    axes[1].set_xlabel('Frequency (MHz)')
    plt.tight_layout()

# *Figure 4: Spectra of Time-Averaged z-Scores*

The average along the time axis of Figures 1 and 3 (though now separated per-polarization). This
plot is useful for showing channels with repeated low-level RFI.

In [ ]:
zscore_spectra()

In [ ]:
def summarize_flagging(flags=flags):
    fig, ax = plt.subplots(figsize=(14,10), dpi=100)
    cmap = matplotlib.colors.ListedColormap(((0, 0, 0),) + matplotlib.colormaps["Set2"].colors[0:2])
    im = ax.imshow(np.where(np.any(~np.isfinite(list(zscore.values())), axis=0), 1, np.where(flags, 2, 0)), 
               aspect='auto', cmap=cmap, interpolation='none', extent=extent)
    im.set_clim([-.5, 2.5])
    cbar = plt.colorbar(im, ax=ax, location='top', aspect=40, pad=.02)
    cbar.set_ticks([0, 1, 2])
    cbar.set_ticklabels(['Unflagged', 'Previously Flagged', 'Flagged Here Using Delayed Filtered z-Scores'])
    ax.set_xlabel('Frequency (MHz)')
    ax.set_ylabel(f'JD - {int(times[0])}')

    # Add LST right axis
    ax2 = ax.twinx()
    ax2.set_ylim(lst_grid[-1], lst_grid[0])
    mod24 = lambda x, _: f"{x % 24:.1f}"
    ax2.yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(mod24))
    ax2.set_ylabel('LST (hours)')

    plt.tight_layout()

# *Figure 5: Summary of Flags Before and After Full-Day Flagging*

This plot shows which times and frequencies were flagged before and after this notebook. It is
directly comparable to Figure 5 of the first round
[full_day_rfi](https://github.com/HERA-Team/hera_notebook_templates/blob/master/notebooks/full_day_rfi.ipynb)
notebook.

In [ ]:
summarize_flagging()

## Save results

In [ ]:
add_to_history = 'by full_day_rfi notebook with the following environment:\n' + '=' * 65 + '\n' + os.popen('conda env export').read() + '=' * 65

In [ ]:
if SAVE_RESULTS:
    # write per-file UVFlag waterfalls built on the zscore files' own time/frequency
    # structure -- no calibration files are read or modified
    tind = 0
    out_flag_files = [zf.replace(RED_AVG_ZSCORE_SUFFIX, FLAG_WATERFALL_SUFFIX) for zf in zscore_files]
    for zf, out_flag_file in zip(zscore_files, out_flag_files):
        uvf_out = UVFlag(zf)
        uvf_out.to_flag()
        uvf_out.flag_array |= flags[tind:tind + len(uvf_out.time_array), :, None]
        uvf_out.history += 'Produced ' + add_to_history
        uvf_out.write(out_flag_file, clobber=True)
        tind += len(uvf_out.time_array)
    print(f'Saved {len(out_flag_files)} *.{FLAG_WATERFALL_SUFFIX} files starting with {out_flag_files[0]}.')

## Metadata

In [ ]:
for repo in ['hera_cal', 'hera_qm', 'hera_filters', 'hera_notebook_templates', 'pyuvdata']:
    exec(f'from {repo} import __version__')
    print(f'{repo}: {__version__}')

In [ ]:
print(f'Finished execution in {(time.time() - tstart) / 60:.2f} minutes.')